# Fig. 3 时间分布（当前去重后的 data/7）
沿用原图统计范围：2000-01-01 至 2020-12-31。每条保留记录计数一次，
不将记录数解释为已确认的独立事故数。三个独立子图分别展示小时、月份和年份分布。
保留现有配色、Times New Roman 字体、10 × 4 英寸画布和 300 dpi PNG；
原始频数不需要误差线。组合图由作者另行制作。


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data" / "7.匹配时间" / "merged_output.csv").is_file())
SOURCE = ROOT / "data" / "7.匹配时间" / "merged_output.csv"
OUTPUT = ROOT / "figure" / "fig3"
QA = ROOT / "检查重复" / "fig3更新_20261108"
OUTPUT.mkdir(exist_ok=True)
QA.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Times New Roman", "DejaVu Sans"],
    "font.size": 10,
    "axes.unicode_minus": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
})
# Only the timestamp is required; read every row without loading unrelated text.
df = pd.read_csv(SOURCE, usecols=["case_dt_hour"], dtype="string")
input_rows = len(df)
print(f"Input records: {input_rows:,}")


# 解析时间

In [ ]:
# Preserve the original inclusive date range; report all exclusions.
timestamps = pd.to_datetime(df["case_dt_hour"], errors="coerce")
invalid_rows = int(timestamps.isna().sum())
start = pd.Timestamp("2000-01-01 00:00:00")
end = pd.Timestamp("2020-12-31 23:59:59")
in_range = timestamps.between(start, end, inclusive="both")
outside_rows = int((timestamps.notna() & ~in_range).sum())
dates = timestamps.loc[in_range]
year_cnt = dates.dt.year.value_counts().reindex(range(2000, 2021), fill_value=0).sort_index()
month_cnt = dates.dt.month.value_counts().reindex(range(1, 13), fill_value=0).sort_index()
hour_cnt = dates.dt.hour.value_counts().reindex(range(24), fill_value=0).sort_index()
assert input_rows == invalid_rows + outside_rows + len(dates)
assert all(int(s.sum()) == len(dates) for s in (hour_cnt, month_cnt, year_cnt))
summary = {
    "source": str(SOURCE.relative_to(ROOT)),
    "input_records": input_rows,
    "invalid_timestamps": invalid_rows,
    "outside_2000_2020": outside_rows,
    "plotted_records": len(dates),
    "date_start": str(start), "date_end": str(end),
    "unit": "retained record, not confirmed unique accident",
    "alignment": "not applicable: three separate single-panel figures",
}
(QA / "statistics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
counts = pd.concat([
    pd.DataFrame({"panel": key, "category": s.index, "count": s.values})
    for key, s in (("hour", hour_cnt), ("month", month_cnt), ("year", year_cnt))
], ignore_index=True)
counts.to_csv(QA / "counts.csv", index=False, encoding="utf-8-sig")
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
year_cnt

# 绘图
仅输出三个独立子图。PDF 留在检查目录用于字体和重叠检查。


In [ ]:
def draw_panel(counts, color, title, xlabel, filename, labels=None, rotation=0):
    x = counts.index.tolist()
    y = counts.values.tolist()
    fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
    ax.bar(x, y, color=color)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    # Avoid grid strokes crossing the exact count labels.
    ax.grid(False)
    ax.set_ylim(0, max(y) * 1.12)
    for xi, yi in zip(x, y):
        ax.annotate(f"{yi:,}", (xi, yi), xytext=(0, 2),
                    textcoords="offset points", ha="center", va="bottom",
                    fontsize=8 if len(x) == 24 else 9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels if labels is not None else [str(v) for v in x],
                       rotation=rotation, rotation_mode="anchor",
                       ha="right" if rotation else "center")
    fig.tight_layout()
    fig.savefig(OUTPUT / f"{filename}.png", dpi=300, bbox_inches="tight", pad_inches=0.1)
    fig.savefig(QA / f"{filename}.pdf", bbox_inches="tight", pad_inches=0.1)
    fig.savefig(QA / f"{filename}.svg", bbox_inches="tight", pad_inches=0.1)
    plt.show()
    plt.close(fig)

draw_panel(hour_cnt, "#7098d8", "(a) Hour Distribution", "Hour (0-23)", "time_hour")


In [ ]:
month_labels = ["Jan.", "Feb.", "Mar.", "Apr.", "May.", "Jun.",
                "Jul.", "Aug.", "Sep.", "Oct.", "Nov.", "Dec."]
draw_panel(month_cnt, "#73afb2", "(b) Month Distribution", "Month", "time_month", labels=month_labels)


In [ ]:
draw_panel(year_cnt, "#e6aeaf", "(c) Year Distribution", "Year", "time_year", rotation=45)
